# Module 10: Panel and Hierarchical Time Series

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

Every module so far has fitted one series at a time. A public safety dataset is
not one series: WADEPS holds roughly three hundred agencies, each with its own
level, its own reporting habits, and the same statewide conditions acting on
all of them.

This module does two things that only become possible once the series are
modelled together. It estimates a common effect far more precisely than any
single agency can, and it keeps a set of published numbers internally
consistent.

**About 40 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

GITHUB = "https://raw.githubusercontent.com/OWNER/REPO/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

monthly = pd.read_csv(BASE + "agency_monthly.csv")
final = monthly[monthly["provisional"] == 0]          # never fit on unfinished months


CALENDAR = pd.period_range("2019-01", "2026-04", freq="M").to_timestamp()


def counts(agency_id):
    """Monthly counts on a complete calendar, so a gap stays visible as missing."""
    d = final[final["agency_id"] == agency_id].sort_values("year_month")
    s = pd.Series(d["n_uof"].values, dtype=float,
                  index=pd.PeriodIndex(d["year_month"], freq="M").to_timestamp())
    s = s.reindex(CALENDAR)
    s.index.freq = "MS"
    return s


def rate(agency_id):
    d = final[final["agency_id"] == agency_id].sort_values("year_month")
    s = pd.Series((100 * d["n_uof"] / d["n_arrests"]).values,
                  index=pd.PeriodIndex(d["year_month"], freq="M").to_timestamp())
    s = s.reindex(CALENDAR)
    s.index.freq = "MS"
    return s


print(f"{final['agency_id'].nunique()} agencies, {final['year_month'].nunique()} months")

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

TREATED = ["A001", "A002", "A004", "A007", "A010"]

f = final.copy()
f = f[~((f["agency_id"] == "A002") & (f["year_month"] == "2021-06"))]   # documented unrest
f["treated"] = f["agency_id"].isin(TREATED).astype(int)
f["settled"] = ((f["treated"] == 1) & (f["year_month"] >= "2023-11")).astype(int)
f["phasein"] = ((f["treated"] == 1) & (f["year_month"] >= "2023-07")
                & (f["year_month"] < "2023-11")).astype(int)
f["lo"] = np.log(f["n_arrests"])

FORM = "n_uof ~ C(agency_id) + C(year_month) + settled + phasein"
pct = lambda b: 100 * (np.exp(b) - 1)


def poisson(formula, data):
    return smf.glm(formula, data, family=sm.families.Poisson(),
                   offset=data["lo"]).fit()


clean = f[f["agency_id"] != "A007"]        # the pre trend violator, see Module 11
print(f"{f['agency_id'].nunique()} agencies, {len(f)} agency months")

## 2. One agency at a time throws information away

The five treated agencies all received the same programme with the same true
effect. Estimate it separately at each of four of them, each against the same
set of controls.

In [ ]:
rows = []
for aid, nm in [("A001", "Stonewick"), ("A002", "Tarnbridge"),
                ("A004", "Millgate"), ("A010", "Pinecrest")]:
    sub = f[((f["agency_id"] == aid) | (f["treated"] == 0)) & (f["agency_id"] != "A007")]
    z = poisson(FORM, sub)
    lo, hi = z.conf_int().loc["settled"]
    rows.append({"agency": nm, "estimate": f"{pct(z.params['settled']):+.1f}%",
                 "95 percent interval": f"[{pct(lo):+.1f}, {pct(hi):+.1f}]"})
z = poisson(FORM, clean)
lo, hi = z.conf_int().loc["settled"]
rows.append({"agency": "ALL FOUR POOLED", "estimate": f"{pct(z.params['settled']):+.1f}%",
             "95 percent interval": f"[{pct(lo):+.1f}, {pct(hi):+.1f}]"})
pd.DataFrame(rows).set_index("agency")

Four estimates of one number, spanning eleven percentage points, from
agencies where the planted effect is **identical**. The entire spread is noise.

Pooled, the estimate lands at 12.6 percent against a truth of 12.0, and the
interval is narrower than any of the four.

## 3. What each set of fixed effects is doing

`C(agency_id)` gives every agency its own baseline rate. `C(year_month)` gives
every month its own level, shared across agencies.

The second one is the important one and it is the one people leave out.

In [ ]:
specs = [
    ("neither set of fixed effects", "n_uof ~ settled + phasein", clean),
    ("agency effects only",          "n_uof ~ C(agency_id) + settled + phasein", clean),
    ("agency and month effects",     FORM, clean),
    ("agency and month, A007 left in", FORM, f),
]
out = []
for name, formula, data in specs:
    z = poisson(formula, data)
    lo, hi = z.conf_int().loc["settled"]
    out.append({"specification": name,
                "settled effect": f"{pct(z.params['settled']):+.1f}%",
                "interval": f"[{pct(lo):+.1f}, {pct(hi):+.1f}]",
                "phase in": f"{pct(z.params['phasein']):+.1f}%",
                "dispersion": round(float(z.pearson_chi2 / z.df_resid), 2)})
print("the truth: settled -12.0 percent, phase in somewhere between -12 and 0\n")
pd.DataFrame(out).set_index("specification")

Read the table from the right.

**The phase in column is the tell.** The first specification reports a phase in
effect of +15.2 percent: the programme supposedly *raised* the use of force
rate during the months it was being introduced, and then cut it. That is not a
finding, it is a symptom. With no month effects, the secular decline common to
every agency has nowhere to go, and the two programme indicators absorb pieces
of it in opposite directions.

This is the same lesson as the exposure coefficient in
[Module 6](../Module_06_Regression_With_ARMA_Errors.md): **the coefficient you
were not interested in is the one that tells you the model is wrong.**

Now read the settled column. Agency effects **alone** give 29.5 percent, more
than twice the truth and worse than using no fixed effects at all. Removing
level differences without removing the common time path leaves the settled
dummy standing in for four years of statewide decline. Half a job is worse than
none here.

## 4. Partial pooling, and a result worth reading carefully

Complete pooling says one number fits all agencies. No pooling says each gets
its own. Partial pooling sits between: each agency's estimate is pulled toward
the common mean by an amount that depends on how noisy it is.

In [ ]:
own = []
for aid in ["A001", "A002", "A004", "A010"]:
    sub = f[((f["agency_id"] == aid) | (f["treated"] == 0)) & (f["agency_id"] != "A007")]
    z = poisson(FORM, sub)
    own.append((aid, z.params["settled"], z.bse["settled"]))

b = np.array([x[1] for x in own])
se = np.array([x[2] for x in own])
pooled = poisson(FORM, clean).params["settled"]

tau2 = max(0.0, b.var(ddof=1) - np.mean(se ** 2))
print(f"  spread of the four estimates      {b.var(ddof=1):.5f}")
print(f"  spread expected from noise alone  {np.mean(se ** 2):.5f}")
print(f"  what is left for real differences {tau2:.5f}\n")
for (aid, bi, si) in own:
    w = tau2 / (tau2 + si ** 2)
    print(f"  {aid}  own {pct(bi):+6.1f}%   weight on its own data {w:.2f}   "
          f"shrunk to {pct(w * bi + (1 - w) * pooled):+6.1f}%")

The four estimates vary **less** than pure sampling noise would predict, so the
between agency variance is estimated at zero and every agency shrinks all the
way to the common value.

That is the correct answer here: the effect really was identical at all five
agencies, and the model has recovered that. But note what the procedure did not
do. It did not prove the effects are the same; it reported that four noisy
estimates give no evidence they differ. **With four agencies, only a very large
difference would have been visible.** Do not read full shrinkage as
homogeneity established.

## 5. Standard errors in a panel

Observations within one agency are correlated over time, so the usual advice is
to cluster standard errors by agency.

In [ ]:
m1 = poisson(FORM, clean)
m2 = smf.glm(FORM, clean, family=sm.families.Poisson(), offset=clean["lo"]).fit(
    cov_type="cluster", cov_kwds={"groups": clean["agency_id"]})
for name, z in [("model based", m1), ("clustered by agency", m2)]:
    lo, hi = z.conf_int().loc["settled"]
    print(f"  {name:22s} se {z.bse['settled']:.4f}   "
          f"[{pct(lo):+.1f}, {pct(hi):+.1f}]")
print(f"\n  number of clusters: {clean['agency_id'].nunique()}")

The two agree to four decimal places, which is worth understanding rather than
celebrating. The month fixed effects have already absorbed the shocks common to
all agencies, which is the main thing clustering protects against, so there is
little correlation left for the cluster correction to find.

**Note the cluster count.** Eleven is far below the forty or so at which
cluster robust standard errors become reliable; with a handful of clusters they
are known to be too small. They agree with the model based errors here, so
nothing turns on it, but in a study with a dozen agencies clustering is not the
safeguard it is often taken for.

## 6. Hierarchies: making the parts sum to the whole

Agencies are one hierarchy. Incident categories are another: eight types that
add up to the total calls for service, each of which splits further into
details.

Forecast the pieces independently and they will not add up.

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

prov = set(monthly[monthly["provisional"] == 1]["year_month"])
bytype = pd.read_csv(BASE + "cfs_monthly_by_type.csv")
bytype = bytype[~bytype["year_month"].isin(prov)]

w = (bytype[bytype["agency_id"] == "A001"]
     .pivot(index="year_month", columns="incident_type", values="n_calls").sort_index())
w.index = pd.PeriodIndex(w.index, freq="M").to_timestamp()
w.index.freq = "MS"
parts = list(w.columns)
w["TOTAL"] = w[parts].sum(axis=1)

H = 12
train, test = w.iloc[:-H], w.iloc[-H:]


def forecast(s):
    m = SARIMAX(np.log(s), order=(0, 1, 1), seasonal_order=(0, 1, 1, 12)).fit(disp=False)
    return np.exp(m.forecast(H).values)


base = {c: forecast(train[c]) for c in w.columns}
bottom_up = np.sum([base[c] for c in parts], axis=0)
top_down = base["TOTAL"]

gap = 100 * (bottom_up - top_down) / top_down
print(f"  forecast of the total, first month : {top_down[0]:,.0f} calls")
print(f"  the eight parts added up           : {bottom_up[0]:,.0f} calls")
print(f"  they disagree by                   : {bottom_up[0] - top_down[0]:,.0f} calls")
print(f"\n  across the 12 months: average {gap.mean():+.2f}%, largest {np.abs(gap).max():.2f}%")

Nobody chose this. Eight sensible forecasts and one sensible forecast of their
total simply do not agree, and if both appear in the same report a reader will
find the discrepancy.

**Reconciliation** adjusts every forecast by the smallest amount that makes
them add up.

In [ ]:
n = len(parts)
S = np.vstack([np.ones((1, n)), np.eye(n)])          # total on top, then the parts
yhat = np.vstack([top_down] + [base[c] for c in parts])
G = np.linalg.inv(S.T @ S) @ S.T
rec = S @ (G @ yhat)

actual = test["TOTAL"].values
mae = lambda v: np.mean(np.abs(v - actual))
print("  MAE on the total, 12 held out months")
print(f"    top down,  forecast the total      {mae(top_down):7.1f}")
print(f"    bottom up, add the eight parts     {mae(bottom_up):7.1f}")
print(f"    reconciled                         {mae(rec[0]):7.1f}")

for label, mat in [("bottom up", np.vstack([base[c] for c in parts])),
                   ("reconciled", rec[1:])]:
    e = np.mean([np.mean(np.abs(mat[i] - test[c].values)) for i, c in enumerate(parts)])
    print(f"  mean MAE across the eight parts, {label:11s} {e:7.1f}")
print(f"\n  reconciled forecasts add up: "
      f"{np.allclose(rec[0], rec[1:].sum(axis=0))}")

**Reconciliation bought coherence, not accuracy.** The total is forecast almost
exactly as well as before and the parts a hair better. That is the honest
result and it is the usual one on data like this.

Coherence is still worth having. It is a requirement of any published table,
and it is free.

## 7. The reason to model the hierarchy, not just the total

Havenbrook changed how it classified calls in January 2023. Look at what that
does to the total, and to the parts.

In [ ]:
a = (bytype[bytype["agency_id"] == "A003"]
     .pivot(index="year_month", columns="incident_type", values="n_calls").sort_index())
before = a[a.index < "2023-01"].mean()
after = a[a.index >= "2023-01"].mean()
tab = pd.DataFrame({"before 2023": before.round(0), "from 2023": after.round(0)})
tab["change"] = (100 * (after - before) / before).round(1).astype(str) + "%"
tab.loc["TOTAL"] = [before.sum().round(0), after.sum().round(0),
                    f"{100 * (after.sum() - before.sum()) / before.sum():.1f}%"]
tab

The total grew 6.0 percent, which is unremarkable next to the 4 to 8 percent
that most categories grew. **Nothing in the aggregate suggests anything
happened.**

Inside it, public order offences rose 60 percent and Other fell 26 percent. It
is a reclassification, and it is completely invisible in the series most
dashboards display.

Any trend, forecast or comparison involving Havenbrook's public order category
across January 2023 is comparing two different definitions.
[Module 12](Module_12_Structural_Breaks.ipynb) is about finding breaks like
this when nobody tells you the date.

## Exercise

The dataset's answer key says a difference in differences estimate recovers
the truth **when A007 is excluded**. Section 3 showed that leaving it in gives
17.0 percent. Find out why, by comparing pre programme trends.

In [ ]:
# Fill in the blank, then run.
COMPARE_PRE_TRENDS = None       # try True

if COMPARE_PRE_TRENDS:
    pre = f[f["year_month"] < "2023-07"].copy()
    idx = pd.PeriodIndex(pre["year_month"], freq="M")
    pre["yr"] = idx.year.values + (idx.month.values - 1) / 12.0
    for label, sub in [("A007 alone", pre[pre["agency_id"] == "A007"]),
                       ("the other treated", pre[(pre["treated"] == 1)
                                                 & (pre["agency_id"] != "A007")]),
                       ("the controls", pre[pre["treated"] == 0])]:
        z = smf.glm("n_uof ~ yr", sub, family=sm.families.Poisson(),
                    offset=sub["lo"]).fit()
        lo, hi = z.conf_int().loc["yr"]
        print(f"  {label:20s} {pct(z.params['yr']):+6.2f}% a year   "
              f"[{pct(lo):+6.2f}, {pct(hi):+6.2f}]")
else:
    print("Set COMPARE_PRE_TRENDS above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
COMPARE_PRE_TRENDS = True
```

Before the programme started, A007 was already falling at **11.96 percent a
year**, interval [15.43, 8.36] down. The other treated agencies fell at 5.17
percent and the controls at 4.46. A007's interval does not come close to
overlapping the controls'.

A difference in differences estimate assumes the treated group would have
followed the control group's path. A007 would not have: it was on a steeper
path of its own, for reasons that predate the training by four years. Keeping
it in hands the settled coefficient four years of A007's private decline and
the estimate moves from 12.6 percent to 17.0.

**No amount of fixed effects fixes this.** Agency effects remove level
differences, month effects remove common time movements, and neither removes an
agency specific *slope*. The diagnosis comes from the pre period, and it is the
one check that must be run before a difference in differences result is
believed.

[Module 11](Module_11_Interrupted_Time_Series.ipynb) takes this apart properly.

</details>

---

**Next:** [Module 11: Interrupted Time Series Done Properly](Module_11_Interrupted_Time_Series.ipynb).

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*